# HUMBUG × 因果推断 (IV / DID / RDD) 蒙特卡洛模拟实验
# HUMBUG × Causal Inference (IV / DID / RDD) Monte Carlo Simulation

**研究问题 / Research question**: Shaikh (1974) 的 HUMBUG 批判表明, 即使 (ln K/L, ln Y/L) 数据在平面上拼出
"HUMBUG" 六个字母, 截面生产函数回归仍可能给出漂亮的拟合。本实验把这一批判扩展到现代因果
推断方法: IV、DID、RDD 各自识别什么参数? "识别成功"是否等于识别了"资本边际产出"这一理论构造?

Shaikh's (1974) HUMBUG critique shows that even when the (ln K/L, ln Y/L) data spell out
the six letters "HUMBUG" on the plane, a cross-sectional production-function regression
can still look well fitted. This experiment extends the critique to modern causal-inference
methods: what parameter does IV / DID / RDD each identify? Does "successful identification"
mean identification of the theoretical construct "marginal product of capital"?

**运行说明 (Anaconda Jupyter) / Run instructions (Anaconda Jupyter)**

| 项目 / Item | 说明 / Description |
| --- | --- |
| 依赖 / Dependencies | `numpy` / `scipy` / `matplotlib` / `pandas` —— Anaconda 发行版自带, 无需额外安装 / bundled with Anaconda, no extra install needed |
| 耗时 / Runtime | 全量 (MC = 500 × 3 模式) 约 1–2 分钟; 快速自检可先把 `CONFIG["mc_reps"]` 改为 `20` / full run (MC = 500 × 3 modes) takes ~1–2 min; for a quick check set `CONFIG["mc_reps"]` to `20` first |
| 输出 / Output | 工作目录内的 `humbug_output/`: `results.json`、`mc_raw.csv`、`*.png`、`MC汇总表_notebook.xlsx` / inside the working directory `humbug_output/`: `results.json`, `mc_raw.csv`, `*.png`, `MC汇总表_notebook.xlsx` |
| 切换模式 / Switch mode | 修改下方 `CONFIG["focus_mode"]` (单次详图) 与 `CONFIG["modes_to_run"]` (MC 范围) / modify `CONFIG["focus_mode"]` (detailed figures) and `CONFIG["modes_to_run"]` (MC scope) below |
| 字体 / Font | 中文字体已做跨平台适配 (Windows 雅黑 / macOS 苹方 / Linux Noto) / cross-platform font setup for Chinese (Windows YaHei / macOS PingFang / Linux Noto) |

In [ ]:
%matplotlib inline
import json
import math
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from IPython.display import display

try:                                    # 向量化正态 CDF (scipy 优先) / vectorized normal CDF (prefer scipy)
    from scipy.special import ndtr as _ndtr

    def _norm_cdf(x):
        return _ndtr(np.asarray(x, float))
except ImportError:                      # 无 scipy 时退回 math.erf / fall back to math.erf when scipy is absent
    def _norm_cdf(x):
        x = np.asarray(x, float)
        flat = x.ravel()
        out = np.array([0.5 * (1.0 + math.erf(t / math.sqrt(2.0))) for t in flat])
        return out.reshape(x.shape)

GK = {"black": "#000000", "d60": "#666666", "d40": "#999999",
      "d25": "#BFBFBF", "d12": "#E0E0E0"}


def setup_mpl():
    """英文字体设置: Times New Roman 衬线, 跨平台回退到同族替代字体。
    English font setup: Times New Roman serif, with cross-platform fallback
    to same-family substitutes."""
    for p in ("C:/Windows/Fonts/times.ttf",
              "C:/Windows/Fonts/timesbd.ttf",
              "/System/Library/Fonts/Supplemental/Times New Roman.ttf",
              "/Library/Fonts/Times New Roman.ttf",
              "/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf"):
        if os.path.exists(p):
            try:
                fm.fontManager.addfont(p)
            except Exception:
                pass
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["Times New Roman", "Liberation Serif",
                                  "Nimbus Roman", "DejaVu Serif"]
    plt.rcParams["mathtext.fontset"] = "stix"   # 数学公式与 Times 风格匹配 / mathtext matches Times style
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["font.size"] = 10.0
    plt.rcParams["axes.linewidth"] = 0.8
    plt.rcParams["xtick.direction"] = "out"
    plt.rcParams["ytick.direction"] = "out"

setup_mpl()                             # 英文字体跨平台适配 / cross-platform English font setup (Windows/macOS/Linux)
print("matplotlib 后端:", plt.get_backend(), "| pandas", pd.__version__, "| numpy", np.__version__)

## 1. 参数配置 CONFIG
## 1. Configuration: CONFIG

三种数据给定模式 (`humbug` / `neoclassical` / `identity`)、三组实验参数与蒙特卡洛设置。

Three data-generating modes (`humbug` / `neoclassical` / `identity`), three experiment
parameter blocks, and the Monte Carlo settings.

In [ ]:
CONFIG = {
    # —— 数据给定模式 / data-generating mode ("humbug" | "neoclassical" | "identity") ——
    "mode": "humbug",                       # 当前聚焦模式 / current focus mode (切换数据给定方式 / switch DGP)
    "modes_to_run": ["humbug", "neoclassical", "identity"],
    "focus_mode": "humbug",                 # 详细图表 (图2-4) 所用模式 / mode used by detailed figures (fig 2-4)
    "seed": 2026,

    # —— HUMBUG 字形 / HUMBUG glyph ——
    "n_humbug_points": 2600,

    # —— 共同结构参数 / common structural parameters ——
    "alpha": 0.30,                          # 真实平均资本弹性 / true average capital elasticity (= identity slope r)
    # μ_i 异质性 (仅 humbug 模式) / heterogeneity of μ_i (only in humbug mode):
    # μ = α + a_c·c + a_s·s + a_q·(s²-1) + e
    "hetero": {"a_c": 0.20, "a_s": -0.10, "a_q": 0.15, "sd_e": 0.05},

    # —— 实验一: IV / Experiment 1: IV ——
    "iv": {"n": 800, "pi_lo": 0.15, "pi_hi": 0.55, "pi_c": 0.9,
           "sd_v": 0.05, "sd_e": 0.02, "delta_dirty": 0.08},

    # —— 实验二: DID / Experiment 2: DID ——
    "did": {"n": 400, "T": 4, "post_start": 2, "dk": 0.25, "sd_nu": 0.04,
            "phi": 0.03, "tau": [0.000, 0.012, 0.024, 0.036], "sd_u": 0.02},

    # —— 实验三: RDD / Experiment 3: RDD ——
    "rdd": {"n": 1200, "s_max": 3.0, "gamma": 0.30, "sd_w": 0.03, "sd_e": 0.02,
            "h": 1.0, "h_grid": [0.5, 1.0, 1.5, 2.0],
            "pseudo": [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]},

    # —— 蒙特卡洛 / Monte Carlo ——
    "mc_reps": 500,

    # —— 输出 / output ——
    "out_dir": "/home/z/my-project/download/humbug_causal",
}

MODES = ["humbug", "neoclassical", "identity"]
MODE_CN = {"humbug": "HUMBUG 字形世界", "neoclassical": "新古典世界",
           "identity": "会计恒等式世界"}

# —— Jupyter 版调整: 输出改写到笔记本所在目录的 humbug_output/ (跨平台) ——
# —— Jupyter adaptation: redirect output to humbug_output/ next to the notebook (cross-platform) ——
CONFIG["out_dir"] = "humbug_output"
os.makedirs(CONFIG["out_dir"], exist_ok=True)
print("输出目录:", os.path.abspath(CONFIG["out_dir"]))
print("聚焦模式:", CONFIG["focus_mode"], "| 运行模式:", CONFIG["modes_to_run"],
      "| MC 次数:", CONFIG["mc_reps"])

## 2. HUMBUG 字形数据生成
## 2. Generating HUMBUG glyph data

用 matplotlib 矢量字形 (DejaVu Serif Bold) 沿轮廓弧长均匀采样约 2600 个点,
归一化到 (k, q) 平面 —— 即 Shaikh (1974) 的"数据拼出 HUMBUG"给定方式。

Uniformly sample about 2600 points along the arc length of the vector glyph outlines
(matplotlib, DejaVu Serif Bold), then normalize to the (k, q) plane — i.e. Shaikh's
(1974) "the data spell out HUMBUG" data-generating scheme.

In [ ]:
def HUMBUG(n_total=2600):
    """用 matplotlib 矢量字形沿轮廓弧长均匀采样生成 HUMBUG 数据。
    Generate HUMBUG data by uniform arc-length sampling along vector glyph outlines.

    返回 (x, y): (k, q) 平面上的 ~n_total 个点, x∈[2.05, 3.90], y∈[0.60, 0.82],
    在平面上拼出 "HUMBUG" 六个字母 (衬线粗体)。
    Returns (x, y): ~n_total points on the (k, q) plane, x∈[2.05, 3.90], y∈[0.60, 0.82],
    spelling out "HUMBUG" (bold serif) on the plane."""
    fp = FontProperties(family="DejaVu Serif", weight="bold")
    tp = TextPath((0, 0), "HUMBUG", size=1.0, prop=fp)
    # 沿路径弧长均匀采样 / uniform arc-length sampling along the path
    segs = []
    for poly in tp.to_polygons():
        poly = np.asarray(poly)
        d = np.sqrt(((poly[1:] - poly[:-1]) ** 2).sum(axis=1))
        segs.append((poly, np.concatenate([[0], np.cumsum(d)])))
    total_len = sum(s[1][-1] for s in segs)
    pts = []
    for poly, cum in segs:
        n = max(2, int(round(n_total * cum[-1] / total_len)))
        u = np.linspace(0, cum[-1], n)
        x = np.interp(u, cum, poly[:, 0])
        y = np.interp(u, cum, poly[:, 1])
        pts.append(np.column_stack([x, y]))
    P = np.concatenate(pts)
    # 归一化到 (k, q) 平面的合适范围 / normalize into the appropriate (k, q) range
    x, y = P[:, 0], P[:, 1]
    x = 2.05 + 1.85 * (x - x.min()) / (x.max() - x.min())
    y = 0.60 + 0.22 * (y - y.min()) / (y.max() - y.min())
    return x, y


_LETTERS = None


def get_letters():
    """缓存并返回 HUMBUG 字形点 (确定性, 只计算一次)。
    Cache and return HUMBUG glyph points (deterministic, computed once)."""
    global _LETTERS
    if _LETTERS is None:
        _LETTERS = HUMBUG(CONFIG["n_humbug_points"])
    return _LETTERS


# —— 字形速览 / quick glyph preview ——
kx, qy = get_letters()
fig, ax = plt.subplots(figsize=(8.4, 3.0), constrained_layout=True)
ax.scatter(kx, qy, s=3, color=GK["d40"], alpha=0.6, linewidths=0)
ax.set_xlabel(r"$\ln(K/L)$  log capital per capita $k$", fontsize=9.5)
ax.set_ylabel(r"$\ln(Y/L)$  log output per capita $q$", fontsize=9.5)
ax.set_title(
    "HUMBUG glyph data (%d points, uniform arc-length sampling of vector glyphs)"
    % len(kx),
    fontsize=10.5,
)
ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
plt.show()

## 3. 计量基础函数
## 3. Core econometric functions

普通最小二乘 (`ols`)、刚好识别 2SLS 与异方差稳健 sandwich 方差 (`iv_robust`)、
平衡面板双向去均值 (`twoway_demean`, 等价于双向固定效应变换)。

Ordinary least squares (`ols`), just-identified 2SLS with heteroskedasticity-robust
sandwich variance (`iv_robust`), and two-way demeaning for balanced panels
(`twoway_demean`, equivalent to the two-way fixed-effects transformation).

In [ ]:
def ols(y, X):
    """普通最小二乘。X 需自带截距列。返回系数、稳健到异方差的常规SE、t、R²。
    Ordinary least squares. X must include an intercept column. Returns coef,
    conventional SE (heteroskedasticity-robust), t, R²."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    n, k = X.shape
    XtX = X.T @ X
    beta = np.linalg.solve(XtX, X.T @ y)
    resid = y - X @ beta
    dof = n - k
    s2 = float(resid @ resid) / dof
    V = s2 * np.linalg.inv(XtX)
    se = np.sqrt(np.diag(V))
    tss = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - float(resid @ resid) / tss if tss > 0 else float("nan")
    return {"beta": beta, "se": se, "t": beta / se, "r2": r2, "n": n}


def iv_robust(y, X, Z):
    """恰好识别 2SLS, 异方差稳健 sandwich 方差。
    Just-identified 2SLS with heteroskedasticity-robust sandwich variance.

    X: 回归变量 (含外生控制与内生变量); Z: 工具变量 (维度与 X 相同)。
    X: regressors (exogenous controls + endogenous variable); Z: instruments (same shape as X).
    返回 (beta, se, resid)。恰好识别时 β = (Z'X)⁻¹Z'y。
    Returns (beta, se, resid). Under just identification, β = (Z'X)⁻¹Z'y."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    Z = np.asarray(Z, float)
    ZtX = Z.T @ X
    beta = np.linalg.solve(ZtX, Z.T @ y)
    resid = y - X @ beta
    Zr = Z * resid[:, None]
    meat = Zr.T @ Zr
    A = np.linalg.inv(ZtX)
    V = A @ meat @ A.T
    se = np.sqrt(np.diag(V))
    return beta, se, resid


def twoway_demean(xmat):
    """平衡面板双向去均值 (企业内与时期内), 等价于双向固定效应变换。
    Two-way demeaning for balanced panels (within firm and within period),
    equivalent to the two-way fixed-effects transformation."""
    xmat = np.asarray(xmat, float)
    return xmat - xmat.mean(1, keepdims=True) - xmat.mean(0, keepdims=True) + xmat.mean()


## 4. 三种数据给定模式的企业截面生成
## 4. Generating firm cross sections under the three data-generating modes

- **humbug**: (k, q) 从 HUMBUG 字形点抽取, 真实边际响应 μ_i 异质
  (μ = α + a_c·c + a_s·s + a_q·(s²−1) + e);
- **neoclassical**: 真实 Cobb-Douglas 世界, μ_i = 0.30 同质 (理论为真的基准);
- **identity**: 会计恒等式 q = w + r·k 精确成立, 斜率 0.30 是利润率参数。

- **humbug**: (k, q) drawn from the HUMBUG glyph points; true marginal response μ_i is
  heterogeneous (μ = α + a_c·c + a_s·s + a_q·(s²−1) + e);
- **neoclassical**: a true Cobb-Douglas world with homogeneous μ_i = 0.30 (the benchmark
  where the theory is correct);
- **identity**: the accounting identity q = w + r·k holds exactly, and the slope 0.30 is a
  profit-rate parameter.

In [ ]:
def true_mu_curve(mode, s):
    """模拟器视角的真实条件边际产出 E[μ|s] (绘图与基准用)。
    True conditional marginal product E[μ|s] from the simulator's perspective
    (for plotting and benchmarks)."""
    a = CONFIG["alpha"]
    s = np.asarray(s, float)
    if mode == "humbug":
        h = CONFIG["hetero"]
        return a + h["a_s"] * s + h["a_q"] * (s ** 2 - 1.0)
    return np.full_like(s, a)


def make_firms(mode, n, rng):
    """按数据给定模式抽取 n 家企业的初始截面 (k0, q0) 与真实边际响应 μ。
    Draw the initial cross-section (k0, q0) and true marginal response μ for
    n firms according to the data-generating mode.

    - humbug      : 从 HUMBUG 字形点中抽取 (k0, q0); 不可观测异质性
                    c (信贷约束) 与 s (企业规模) 独立抽取;
                    μ = α + a_c·c + a_s·s + a_q·(s²-1) + e, E[μ]=α。
                    (k0, q0) drawn from HUMBUG glyph points; unobserved
                    heterogeneity c (credit constraint) and s (firm size)
                    drawn independently; μ = α + a_c·c + a_s·s + a_q·(s²-1) + e, E[μ]=α.
    - neoclassical: k0~N(2.9,0.35), q0 = -0.20 + α·k0 + ε (真生产函数);
                    μ_i = α 同质。
                    (true production function); μ_i = α homogeneous.
    - identity    : q0 = -0.20 + α·k0 精确成立 (会计恒等式, R²=1);
                    μ_i = α 同质 (恒等式斜率 = 利润率参数)。
                    (identity holds exactly, R²=1); μ_i = α homogeneous
                    (identity slope = profit-rate parameter)."""
    if mode == "humbug":
        kx, qy = get_letters()
        idx = rng.choice(len(kx), size=n, replace=True)
        k0 = kx[idx] + rng.normal(0, 0.005, n)     # 微小抖动, 避免点完全重合 / tiny jitter to avoid exact overlap
        q0 = qy[idx] + rng.normal(0, 0.005, n)
        h = CONFIG["hetero"]
        c = rng.normal(size=n)
        s = np.clip(rng.normal(size=n), -CONFIG["rdd"]["s_max"], CONFIG["rdd"]["s_max"])
        mu = (CONFIG["alpha"] + h["a_c"] * c + h["a_s"] * s
              + h["a_q"] * (s ** 2 - 1.0) + rng.normal(0, h["sd_e"], n))
    elif mode == "neoclassical":
        k0 = rng.normal(2.9, 0.35, n)
        q0 = -0.20 + CONFIG["alpha"] * k0 + rng.normal(0, 0.01, n)
        c = rng.normal(size=n)
        s = np.clip(rng.normal(size=n), -CONFIG["rdd"]["s_max"], CONFIG["rdd"]["s_max"])
        mu = np.full(n, CONFIG["alpha"])
    elif mode == "identity":
        k0 = rng.normal(2.9, 0.35, n)
        q0 = -0.20 + CONFIG["alpha"] * k0           # 恒等式精确成立, 无噪声 / identity holds exactly, no noise
        c = rng.normal(size=n)
        s = np.clip(rng.normal(size=n), -CONFIG["rdd"]["s_max"], CONFIG["rdd"]["s_max"])
        mu = np.full(n, CONFIG["alpha"])
    else:
        raise ValueError("未知数据给定模式 / unknown data-generating mode: %s" % mode)
    return {"k0": k0, "q0": q0, "mu": mu, "c": c, "s": s}


# —— 三种模式的截面速览 (独立种子, 不影响实验随机流) ——
# —— quick cross-section preview of the three modes (independent seed, does not affect the experimental random stream) ——
MODE_EN = {"humbug": "Humbug (glyph data)",
           "neoclassical": "Neoclassical",
           "identity": "Accounting identity"}

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.9), constrained_layout=True)
for ax, mode in zip(axes, MODES):
    rng = np.random.default_rng([CONFIG["seed"], 42])
    fb = make_firms(mode, 1200, rng)
    ax.scatter(fb["k0"], fb["q0"], s=3, color=GK["d40"], alpha=0.55, linewidths=0)
    ax.set_title(MODE_EN[mode], fontsize=10.5)
    ax.set_xlabel(r"$k$", fontsize=9.5)
    ax.set_ylabel(r"$q$", fontsize=9.5)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
plt.show()

## 5. 实验一: 工具变量 (IV)
## 5. Experiment 1: Instrumental variables (IV)

干净/脏工具**共享同一组冲击**, 只有直接效应 δ 不同 (δ = 0.08 经融资成本/
需求/技术渠道违背排除限制)。合规强度 π(c) = 0.15 + 0.55Φ(0.9c) 单调递增。

恰好识别 2SLS 的概率极限: 干净工具 → E[μπ]/E[π] (合规者加权平均, LATE 型参数);
脏工具 → (E[μπ] + δ)/E[π]。两者都不等于总体平均 μ_pop = E[μ]。

The clean and dirty instruments **share the same shocks**; only the direct effect δ
differs (δ = 0.08 violates exclusion through financing-cost / demand / technology
channels). Compliance strength π(c) = 0.15 + 0.55Φ(0.9c) is monotonically increasing.

Probability limits of just-identified 2SLS: clean instrument → E[μπ]/E[π]
(complier-weighted average, a LATE-type parameter); dirty instrument →
(E[μπ] + δ)/E[π]. Neither equals the population mean μ_pop = E[μ].

In [ ]:
def iv_simulate(firms, rng):
    """抽取 IV 实验的随机冲击 (Z, v, e) 与企业合规强度 π(c)。
    Draw random shocks (Z, v, e) and firm compliance strength π(c) for the IV experiment.

    模型 / Model:
      第一阶段 / first stage   k_i = k0_i + π_i·Z_i + v_i,   π_i = pi_lo + pi_hi·Φ(pi_c·c_i) > 0 (单调 / monotone)
      结构方程 / structural    q_i = q0_i + μ_i·Δk_i + δ·Z_i + e_i
    脏工具 arm 下 δ = delta_dirty: Z 经融资成本/需求/技术渠道直接影响 q,
    违背排除限制; 干净 arm 下 δ = 0。
    In the dirty arm, δ = delta_dirty: Z directly affects q through financing
    cost / demand / technology channels, violating exclusion; in the clean arm δ = 0."""
    iv = CONFIG["iv"]
    n = len(firms["k0"])
    Z = rng.normal(size=n)
    v = rng.normal(0, iv["sd_v"], n)
    e = rng.normal(0, iv["sd_e"], n)
    pi = iv["pi_lo"] + iv["pi_hi"] * _norm_cdf(iv["pi_c"] * firms["c"])
    return {"Z": Z, "v": v, "e": e, "pi": pi}


def iv_estimate(firms, shock, arm):
    """在给定冲击下估计 OLS 与 2SLS, 并计算模拟器视角的三重基准。
    Estimate OLS and 2SLS under the given shocks, and compute the three
    simulator-side benchmarks.

    恰好识别 2SLS 的概率极限 (Wald 估计量) / probability limit of just-identified 2SLS (Wald estimator):
        clean: β → E[μπ]/E[π]  (合规者加权平均, LATE 型参数 / complier-weighted mean, LATE-type parameter)
        dirty: β → (E[μπ] + δ)/E[π]  (混入直接渠道, 排除限制被违背 / mixes in direct channel, exclusion violated)
    两面都不等于总体平均 μ_pop = E[μ] —— 这就是"因果参数与理论构念的解释鸿沟"。
    Neither equals the population mean μ_pop = E[μ] — the interpretational gap
    between causal parameters and the theoretical construct."""
    iv = CONFIG["iv"]
    Z, v, e, pi = shock["Z"], shock["v"], shock["e"], shock["pi"]
    dk = pi * Z + v
    k = firms["k0"] + dk
    delta = iv["delta_dirty"] if arm == "dirty" else 0.0
    q = firms["q0"] + firms["mu"] * dk + delta * Z + e

    one = np.ones_like(k)
    m_ols = ols(q, np.column_stack([one, k]))
    fs = ols(k, np.column_stack([one, Z]))
    F = float(fs["t"][1] ** 2)
    beta, se, _ = iv_robust(q, np.column_stack([one, k]),
                            np.column_stack([one, Z]))

    mu = firms["mu"]
    n = len(mu)
    mu_pop = float(mu.mean())
    mu_comp = float((mu * pi).sum() / pi.sum())
    estimand = float(((mu * pi).sum() / n + delta) / (pi.sum() / n))
    return {
        "arm": arm, "delta": delta,
        "pi_hat": float(fs["beta"][1]), "pi_se": float(fs["se"][1]),
        "first_stage_F": F,
        "ols_beta": float(m_ols["beta"][1]), "ols_se": float(m_ols["se"][1]),
        "beta": float(beta[1]), "se": float(se[1]),
        "mu_pop": mu_pop, "mu_compliers": mu_comp, "estimand": estimand,
        "n": n,
    }


## 6. 实验二: 双重差分 (DID)
## 6. Experiment 2: Difference-in-differences (DID)

小企业 (s < 0) 投资补贴面板 (400 企业 × 4 期), 平行趋势按构造成立 (共同时间效应 τ_t)。
政策含直接渠道 φ = 0.03, 因此 DID 识别的政策总效应 θ = E[μ|Treat]·Δk + φ;
以 θ/ΔK 解读为"资本边际产出"得到的是 E[μ|Treat] + φ/Δk —— 既非 μ_pop, 也非处理组平均。
另含安慰剂检验 (只用前两期, 第 1 期作伪处理期)。

Investment-subsidy panel for small firms (s < 0): 400 firms × 4 periods, with parallel
trends holding by construction (common time effects τ_t). The policy includes a direct
channel φ = 0.03, so DID identifies the total policy effect θ = E[μ|Treat]·Δk + φ;
reading θ/ΔK as the "marginal product of capital" yields E[μ|Treat] + φ/Δk — neither
μ_pop nor the treated-group mean. Also includes a placebo test (first two periods only,
with period 1 as a fake post-period).

In [ ]:
def did_simulate(firms, rng):
    """生成 DID 面板 (n×T), 处理组为小企业 (s<0), post 为后两期。
    Generate the DID panel (n×T): treated group = small firms (s<0), post = last two periods.

    模型 / Model:
      k_it = k0_i + (dk + ν_it)·D_it,                D_it = Treat_i × Post_t
      q_it = q0_i + τ_t + μ_i·(dk + ν_it)·D_it + φ·D_it + u_it
    平行趋势按构造成立 (共同时间效应 τ_t); φ 汇总融资约束放松、技术选择
    变化与市场竞争地位三条直接渠道 —— 政策总效应 ≠ μ·Δk。
    Parallel trends hold by construction (common time effects τ_t); φ aggregates
    three direct channels (relaxed financing constraints, technology choice,
    market position) — total policy effect ≠ μ·Δk."""
    d = CONFIG["did"]
    n, T = d["n"], d["T"]
    treat = firms["s"] < 0.0
    post = np.arange(T) >= d["post_start"]
    D = (treat[:, None] & post[None, :]).astype(float)
    nu = rng.normal(0, d["sd_nu"], (n, T))
    dk = np.where(D == 1.0, d["dk"] + nu, 0.0)
    k = firms["k0"][:, None] + dk
    q = (firms["q0"][:, None] + np.asarray(d["tau"])[None, :]
         + firms["mu"][:, None] * dk + d["phi"] * D
         + rng.normal(0, d["sd_u"], (n, T)))
    return {"k": k, "q": q, "D": D, "treat": treat, "post": post, "n": n, "T": T,
            "mu": firms["mu"]}


def did_estimate(panel):
    """TWFE 双重差分: ATT (θ)、资本 DID (ΔK)、隐含边际产出 (θ/ΔK) 与 FE 生产回归。
    TWFE DID: ATT (θ), capital DID (ΔK), implied marginal product (θ/ΔK),
    and the FE production regression.

    DID 识别的是政策总效应 / DID identifies the total policy effect:
    θ = E[μ|Treat]·Δk + φ;
    研究者若以 θ/ΔK 解读为"资本边际产出", 得到 E[μ|Treat] + φ/Δk,
    既非总体平均 μ_pop, 也非处理组真实平均 —— 识别成功 ≠ 理论构念。
    Reading θ/ΔK as the marginal product of capital yields E[μ|Treat] + φ/Δk,
    which is neither the population mean μ_pop nor the true treated-group mean —
    successful identification ≠ the theoretical construct."""
    d = CONFIG["did"]
    k, q, D = panel["k"], panel["q"], panel["D"]
    n, T = panel["n"], panel["T"]
    qd = twoway_demean(q).ravel()
    Dd = twoway_demean(D).ravel()
    kd = twoway_demean(k).ravel()

    sxx = float((Dd * Dd).sum())
    theta = float((qd * Dd).sum() / sxx)
    resid = qd - theta * Dd
    dof = n * T - n - T
    s2 = float(resid @ resid) / dof
    se_theta = math.sqrt(s2 / sxx)

    sxxk = float((kd * kd).sum())
    fe = float((qd * kd).sum() / sxxk)
    residk = qd - fe * kd
    se_fe = math.sqrt(float(residk @ residk) / dof / sxxk)

    tr, po = panel["treat"], panel["post"]
    dk_did = ((k[tr][:, po].mean() - k[tr][:, ~po].mean())
              - (k[~tr][:, po].mean() - k[~tr][:, ~po].mean()))

    mu_treated = float(panel["mu"][tr].mean())
    implied_true = mu_treated + d["phi"] / d["dk"]
    return {
        "theta": theta, "se_theta": se_theta,
        "dk_did": float(dk_did),
        "implied": theta / dk_did,
        "implied_se": se_theta / dk_did,
        "fe": fe, "se_fe": se_fe,
        "mu_treated": mu_treated, "implied_true": implied_true,
        "mu_pop": float(panel["mu"].mean()),
        "share_treated": float(tr.mean()), "n": n, "T": T,
    }


def did_placebo(panel):
    """安慰剂检验: 只用前两期, 把第 1 期当伪处理期, 检验前趋势平行。
    Placebo test: use only the first two periods, treat period 1 as a fake
    post-period, and test whether pre-trends are parallel."""
    sl = slice(0, 2)
    k, q = panel["k"][:, sl], panel["q"][:, sl]
    n = panel["n"]
    fake_post = np.array([False, True])
    D = (panel["treat"][:, None] & fake_post[None, :]).astype(float)
    qd = twoway_demean(q).ravel()
    Dd = twoway_demean(D).ravel()
    sxx = float((Dd * Dd).sum())
    theta = float((qd * Dd).sum() / sxx)
    resid = qd - theta * Dd
    dof = 2 * n - n - 2
    s2 = float(resid @ resid) / dof
    return {"theta": theta, "se": math.sqrt(s2 / sxx)}


## 7. 实验三: 断点回归 (RDD)
## 7. Experiment 3: Regression discontinuity design (RDD)

企业规模门槛 s* = 0 的补贴 (sharp), 资本在门槛处跳升 γ = 0.30, 连续性假设按构造成立。
局部线性 2SLS (窗口 |s−c| ≤ h, 两侧斜率允许不同) 识别门槛附近企业的局部效应,
概率极限为 E[μ|s=c]。伪门槛扫描 (s* ∈ [−2, 2], 复用同一组 w, e) 描出真实异质性曲线。

A sharp subsidy at the firm-size threshold s* = 0, with capital jumping by γ = 0.30 at
the cutoff and continuity holding by construction. Local-linear 2SLS (window |s−c| ≤ h,
with side-specific slopes) identifies the local effect of firms near the threshold, with
probability limit E[μ|s=c]. A pseudo-cutoff scan (s* ∈ [−2, 2], reusing the same w, e)
traces out the true heterogeneity curve.

In [ ]:
def rdd_simulate(firms, rng, cutoff=0.0, w=None, e=None):
    """在给定门槛 cutoff 处施加补贴政策, 生成 RDD 截面数据。
    Apply the subsidy policy at a given cutoff and generate RDD cross-section data.

    模型 / Model:
      D_i = 1{s_i ≥ cutoff}  (规模门槛, sharp / size threshold, sharp)
      k_i = k0_i + γ·D_i + w_i        (资本在门槛处跳升 γ / capital jumps by γ at the cutoff)
      q_i = q0_i + μ_i·(γ·D_i + w_i) + e_i
    无直接效应 → 连续性假设按构造成立; 识别的是门槛附近企业的局部效应。
    伪门槛扫描时复用同一组 (w, e), 只改 cutoff —— 相当于政策门槛的反事实平移。
    No direct effect → continuity holds by construction; identifies the local
    effect of firms near the cutoff. Pseudo-cutoff scans reuse the same (w, e)
    and only change the cutoff — a counterfactual shift of the policy threshold."""
    r = CONFIG["rdd"]
    n = len(firms["s"])
    if w is None:
        w = rng.normal(0, r["sd_w"], n)
    if e is None:
        e = rng.normal(0, r["sd_e"], n)
    D = (firms["s"] >= cutoff).astype(float)
    dk = r["gamma"] * D + w
    k = firms["k0"] + dk
    q = firms["q0"] + firms["mu"] * dk + e
    return {"s": firms["s"], "k0": firms["k0"], "q0": firms["q0"],
            "mu": firms["mu"], "D": D, "k": k, "q": q, "w": w, "e": e,
            "cutoff": cutoff}


def rdd_estimate(data, h):
    """模糊 RDD 局部线性 2SLS: 窗口 |s-c|≤h 内, 用 D 作 k 的工具。
    Fuzzy RDD local-linear 2SLS: within window |s-c|≤h, use D as an instrument for k.

    估计方程 (两侧斜率允许不同) / estimating equation (side-specific slopes allowed):
      q = β·k + a + b_l·(s-c)·(1-D) + b_r·(s-c)·D + 误差 / error
      工具 / instruments: [1, (s-c)(1-D), (s-c)D, D]
    β̂ 恒等于两侧局部线性跳跃之比 J_q/J_k, 概率极限为 E[μ|s=c]。
    β̂ equals the ratio J_q/J_k of the two local-linear jumps, with probability
    limit E[μ|s=c]."""
    s, k, q, D = data["s"], data["k"], data["q"], data["D"]
    c = data["cutoff"]
    m = np.abs(s - c) <= h
    nw = int(m.sum())
    sc = s[m] - c
    Dm = D[m]
    one = np.ones(nw)
    exog = np.column_stack([one, sc * (1.0 - Dm), sc * Dm])
    X = np.column_stack([exog, k[m]])
    Z = np.column_stack([exog, Dm])
    beta, se, _ = iv_robust(q[m], X, Z)
    fsk = ols(k[m], np.column_stack([exog, Dm]))
    rfq = ols(q[m], np.column_stack([exog, Dm]))
    return {"beta": float(beta[3]), "se": float(se[3]),
            "Jk": float(fsk["beta"][3]), "Jq": float(rfq["beta"][3]),
            "n_win": nw, "h": h}


def rdd_pseudo_scan(firms, rng, w, e):
    """伪门槛扫描: 把政策门槛平移到每个 s*, 重新施加处理并估计局部效应。
    Pseudo-cutoff scan: shift the policy threshold to each s*, re-apply
    treatment and estimate the local effect.

    估计值随门槛移动而变化, 描出真实异质性 E[μ|s] 曲线 —— 说明 RDD 识别的
    是"门槛处的局部参数", 不存在可供外推的单一"资本边际产出"。
    Estimates vary with the cutoff, tracing out the true heterogeneity curve
    E[μ|s] — RDD identifies a "local parameter at the cutoff"; there is no
    single "marginal product of capital" available for extrapolation."""
    out = []
    for c in CONFIG["rdd"]["pseudo"]:
        data = rdd_simulate(firms, rng, cutoff=c, w=w, e=e)
        est = rdd_estimate(data, CONFIG["rdd"]["h"])
        out.append({"cutoff": c, "beta": est["beta"], "se": est["se"],
                    "Jk": est["Jk"], "Jq": est["Jq"]})
    return out


## 8. 单次代表性抽样与蒙特卡洛引擎
## 8. Single representative draw and Monte Carlo engine

`run_headline` 对给定模式跑一次固定种子的代表性抽样 (基准 OLS + 三组实验全部结果);
`run_mc` 做 `mc_reps` 次重复, 汇总每个估计量的分布、相对"总体平均真实边际产出 μ_pop"
与相对"方法自身识别对象 (概率极限)"的**双重偏差**, 以及 95% 置信区间覆盖率。

`run_headline` runs one fixed-seed representative draw for the given mode (baseline OLS +
all results from the three experiments); `run_mc` repeats `mc_reps` times and summarizes
each estimator's distribution, its **double bias** — relative to the population average
true marginal product μ_pop and to the method's own estimand (its probability limit) —
along with 95% confidence-interval coverage.

In [ ]:
def run_headline(mode):
    """对给定模式跑一次代表性抽样 (固定种子), 返回全部单次结果与作图数据。
    Run one representative draw (fixed seed) for the given mode and return all
    single-run results together with the plotting data."""
    rng = np.random.default_rng([CONFIG["seed"], MODES.index(mode), 7])
    res, dbg = {}, {}

    # —— 基准 OLS (无政策/无工具的原始截面) ——
    # —— baseline OLS (raw cross-section without policy/instruments) ——
    fr = make_firms(mode, CONFIG["rdd"]["n"], rng)
    b = ols(fr["q0"], np.column_stack([np.ones(len(fr["q0"])), fr["k0"]]))
    res["baseline"] = {"beta": float(b["beta"][1]), "se": float(b["se"][1]),
                       "r2": float(b["r2"]), "n": int(b["n"])}

    # —— 实验一: IV (干净/脏共享同一组冲击, 只有 δ 不同) ——
    # —— Experiment 1: IV (clean/dirty share the same shocks; only δ differs) ——
    fiv = make_firms(mode, CONFIG["iv"]["n"], rng)
    shock = iv_simulate(fiv, rng)
    res["iv_clean"] = iv_estimate(fiv, shock, "clean")
    res["iv_dirty"] = iv_estimate(fiv, shock, "dirty")
    dbg["iv"] = {"firms": fiv, "shock": shock,
                 "k": fiv["k0"] + shock["pi"] * shock["Z"] + shock["v"],
                 "q_clean": None, "q_dirty": None}

    # —— 实验二: DID ——
    # —— Experiment 2: DID ——
    fd = make_firms(mode, CONFIG["did"]["n"], rng)
    panel = did_simulate(fd, rng)
    res["did"] = did_estimate(panel)
    res["did_placebo"] = did_placebo(panel)
    dbg["did"] = panel

    # —— 实验三: RDD (各带宽/伪门槛复用同一组 w, e) ——
    # —— Experiment 3: RDD (all bandwidths/pseudo-cutoffs reuse the same w, e) ——
    r = CONFIG["rdd"]
    w = rng.normal(0, r["sd_w"], len(fr["s"]))
    e = rng.normal(0, r["sd_e"], len(fr["s"]))
    data0 = rdd_simulate(fr, rng, 0.0, w=w, e=e)
    res["rdd"] = rdd_estimate(data0, r["h"])
    res["rdd_local_true"] = float(true_mu_curve(mode, 0.0))
    res["rdd_h"] = {str(h): rdd_estimate(rdd_simulate(fr, rng, 0.0, w=w, e=e), h)
                    for h in r["h_grid"]}
    res["rdd_pseudo"] = rdd_pseudo_scan(fr, rng, w, e)
    dbg["rdd"] = data0
    dbg["firms_rdd"] = fr

    # —— 外推演示: 用 RDD 局部估计预测"全体企业资本同增 γ"的总效应 ——
    # —— extrapolation demo: use the RDD local estimate to predict the aggregate
    #    effect of "all firms raising capital by γ" ——
    mu_pop = float(fr["mu"].mean())
    g = r["gamma"]
    res["extrapolation"] = {
        "gamma": g,
        "pred": res["rdd"]["beta"] * g,
        "true": mu_pop * g,
        "mu_pop": mu_pop,
        "rel_error": res["rdd"]["beta"] / mu_pop - 1.0,
    }
    return res, dbg


In [ ]:
MC_KEYS = ["ols_base", "ols_base_se", "iv_clean", "iv_clean_se", "iv_F",
           "iv_dirty", "iv_dirty_se", "did_theta", "did_theta_se", "did_dk",
           "did_implied", "did_implied_se", "did_fe", "did_fe_se", "did_placebo",
           "did_placebo_se", "rdd", "rdd_se", "rdd_h05", "rdd_h15", "rdd_h20",
           "mu_pop", "mu_comp", "ivd_estimand", "did_treated_mu",
           "did_implied_true", "rdd_local_true"]


def mc_one(mode, rep):
    """蒙特卡洛单次重复: 重新抽取企业与全部冲击, 估计所有方法并记录。
    One Monte Carlo repetition: redraw firms and all shocks, estimate every
    method, and record the results."""
    rng = np.random.default_rng([CONFIG["seed"] + 1, MODES.index(mode), rep])
    r = CONFIG["rdd"]

    fb = make_firms(mode, r["n"], rng)
    fiv = make_firms(mode, CONFIG["iv"]["n"], rng)
    fd = make_firms(mode, CONFIG["did"]["n"], rng)

    b = ols(fb["q0"], np.column_stack([np.ones(len(fb["q0"])), fb["k0"]]))
    shock = iv_simulate(fiv, rng)
    ivc = iv_estimate(fiv, shock, "clean")
    ivd = iv_estimate(fiv, shock, "dirty")

    panel = did_simulate(fd, rng)
    de = did_estimate(panel)
    pl = did_placebo(panel)

    w = rng.normal(0, r["sd_w"], r["n"])
    e = rng.normal(0, r["sd_e"], r["n"])
    data = rdd_simulate(fb, rng, 0.0, w=w, e=e)
    re0 = rdd_estimate(data, r["h"])

    row = {
        "ols_base": float(b["beta"][1]), "ols_base_se": float(b["se"][1]),
        "iv_clean": ivc["beta"], "iv_clean_se": ivc["se"], "iv_F": ivc["first_stage_F"],
        "iv_dirty": ivd["beta"], "iv_dirty_se": ivd["se"],
        "did_theta": de["theta"], "did_theta_se": de["se_theta"],
        "did_dk": de["dk_did"], "did_implied": de["implied"],
        "did_implied_se": de["implied_se"],
        "did_fe": de["fe"], "did_fe_se": de["se_fe"],
        "did_placebo": pl["theta"], "did_placebo_se": pl["se"],
        "rdd": re0["beta"], "rdd_se": re0["se"],
        "rdd_h05": rdd_estimate(data, 0.5)["beta"],
        "rdd_h15": rdd_estimate(data, 1.5)["beta"],
        "rdd_h20": rdd_estimate(data, 2.0)["beta"],
        "mu_pop": ivc["mu_pop"], "mu_comp": ivc["mu_compliers"],
        "ivd_estimand": ivd["estimand"], "did_treated_mu": de["mu_treated"],
        "did_implied_true": de["implied_true"],
        "rdd_local_true": float(true_mu_curve(mode, 0.0)),
    }
    return row


def run_mc(mode, reps):
    """蒙特卡洛主循环: reps 次重复 → 每个估计量的分布与双重偏差汇总。
    Monte Carlo main loop: reps repetitions → distribution and double-bias
    summary for each estimator."""
    rows = [mc_one(mode, rep) for rep in range(reps)]

    def arr(key):
        return np.array([row[key] for row in rows], float)

    mu_pop = arr("mu_pop").mean()
    estimators = {
        "ols_base": {"est": "ols_base", "se": "ols_base_se", "estimand": "mu_pop"},
        "iv_clean": {"est": "iv_clean", "se": "iv_clean_se", "estimand": "mu_comp"},
        "iv_dirty": {"est": "iv_dirty", "se": "iv_dirty_se", "estimand": "ivd_estimand"},
        "did_implied": {"est": "did_implied", "se": "did_implied_se", "estimand": "did_implied_true"},
        "did_fe": {"est": "did_fe", "se": "did_fe_se", "estimand": "did_implied_true"},
        "rdd": {"est": "rdd", "se": "rdd_se", "estimand": "rdd_local_true"},
    }
    summary = {}
    for name, spec in estimators.items():
        est = arr(spec["est"])
        estand = arr(spec["estimand"]).mean()
        item = {
            "mean": float(est.mean()), "sd": float(est.std(ddof=1)),
            "q025": float(np.percentile(est, 2.5)),
            "q975": float(np.percentile(est, 97.5)),
            "estimand": float(estand),
            "bias_vs_pop": float(est.mean() - mu_pop),
            "bias_vs_estimand": float(est.mean() - estand),
        }
        if spec["se"] is not None:
            se = arr(spec["se"])
            item["cover_pop"] = float(np.mean(np.abs(est - mu_pop) <= 1.96 * se))
        else:
            item["cover_pop"] = None
        summary[name] = item
    summary["_meta"] = {
        "reps": reps, "mu_pop": float(mu_pop),
        "mu_comp": float(arr("mu_comp").mean()),
        "ivd_estimand": float(arr("ivd_estimand").mean()),
        "did_treated_mu": float(arr("did_treated_mu").mean()),
        "did_implied_true": float(arr("did_implied_true").mean()),
        "rdd_local_true": float(arr("rdd_local_true").mean()),
        "iv_F_min": float(arr("iv_F").min()), "iv_F_med": float(np.median(arr("iv_F"))),
        "did_placebo_mean": float(arr("did_placebo").mean()),
        "did_placebo_cover0": float(np.mean(
            np.abs(arr("did_placebo")) <= 1.96 * arr("did_placebo_se"))),
        "rdd_h05_mean": float(arr("rdd_h05").mean()),
        "rdd_h15_mean": float(arr("rdd_h15").mean()),
        "rdd_h20_mean": float(arr("rdd_h20").mean()),
    }
    return summary, rows


In [ ]:
def to_jsonable(obj):
    """把 numpy 类型递归转为可 JSON 序列化的 Python 基本类型。
    Recursively convert numpy types into JSON-serializable Python primitives."""
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return float(obj)
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    return obj


## 9. 运行: 单次代表性抽样
## 9. Run: single representative draw

先看聚焦模式 (默认 HUMBUG 字形世界) 的一次抽样: 各方法点估计 vs 双重基准。

First look at one draw under the focus mode (default: the HUMBUG glyph world): point
estimates from each method vs the two benchmarks.

In [ ]:
MODE = CONFIG["focus_mode"]        # 可改为 "neoclassical" / "identity" 单独查看 / can be set to "neoclassical" / "identity" to inspect individually
res, dbg = run_headline(MODE)

print("== %s: 单次代表性抽样 (种子 %d) ==" % (MODE_CN[MODE], CONFIG["seed"]))
print("基准 OLS : β = %.4f (SE %.4f), R² = %.4f"
      % (res["baseline"]["beta"], res["baseline"]["se"], res["baseline"]["r2"]))
print("IV 干净  : β = %.4f (SE %.4f, 第一阶段 F = %.0f)"
      % (res["iv_clean"]["beta"], res["iv_clean"]["se"], res["iv_clean"]["first_stage_F"]))
print("IV 脏    : β = %.4f (SE %.4f)"
      % (res["iv_dirty"]["beta"], res["iv_dirty"]["se"]))
print("DID      : ATT = %.4f (SE %.4f), ΔK = %.4f, 隐含 MPK = %.4f"
      % (res["did"]["theta"], res["did"]["se_theta"], res["did"]["dk_did"], res["did"]["implied"]))
print("安慰剂   : %.4f (SE %.4f)"
      % (res["did_placebo"]["theta"], res["did_placebo"]["se"]))
print("RDD      : β = %.4f (SE %.4f), 局部真值 = %.4f"
      % (res["rdd"]["beta"], res["rdd"]["se"], res["rdd_local_true"]))

_rows = [
    ("基准 OLS (无政策截面)", res["baseline"]["beta"], res["baseline"]["se"],
     res["extrapolation"]["mu_pop"]),
    ("OLS (IV 实验样本)", res["iv_clean"]["ols_beta"], res["iv_clean"]["ols_se"],
     res["iv_clean"]["mu_pop"]),
    ("2SLS 干净工具", res["iv_clean"]["beta"], res["iv_clean"]["se"],
     res["iv_clean"]["mu_compliers"]),
    ("2SLS 脏工具", res["iv_dirty"]["beta"], res["iv_dirty"]["se"],
     res["iv_dirty"]["estimand"]),
    ("DID 隐含边际产出 (θ/ΔK)", res["did"]["implied"], res["did"]["implied_se"],
     res["did"]["implied_true"]),
    ("FE 生产回归", res["did"]["fe"], res["did"]["se_fe"], res["did"]["implied_true"]),
    ("RDD 局部估计 (s*=0)", res["rdd"]["beta"], res["rdd"]["se"], res["rdd_local_true"]),
]
df_single = pd.DataFrame(_rows, columns=["估计量", "点估计", "SE", "自身识别对象"])
df_single["偏差 (vs μ_pop)"] = df_single["点估计"] - res["extrapolation"]["mu_pop"]
display(df_single.round(4))

## 10. 运行: 蒙特卡洛模拟 (N = 500 × 3 模式)
## 10. Run: Monte Carlo simulation (N = 500 × 3 modes)

对 `modes_to_run` 中的每个模式执行"单次抽样 + 500 次蒙特卡洛",
保存 `results.json` 与 `mc_raw.csv` 到输出目录。

For each mode in `modes_to_run`, run "one representative draw + 500 Monte Carlo
repetitions", and save `results.json` and `mc_raw.csv` to the output directory.

In [ ]:
kx, qy = get_letters()
out = {
    "config": {k: v for k, v in CONFIG.items() if k != "out_dir"},
    "letters": {"n_points": int(len(kx)),
                "k_min": float(kx.min()), "k_max": float(kx.max()),
                "q_min": float(qy.min()), "q_max": float(qy.max()),
                "k_mean": float(kx.mean()), "q_mean": float(qy.mean())},
    "modes": {}, "mc_rows": {},
}
debugs = {}
for mode in CONFIG["modes_to_run"]:
    print("== 模式: %s ==" % mode)
    head, dbg = run_headline(mode)
    debugs[mode] = dbg
    mcsum, rows = run_mc(mode, CONFIG["mc_reps"])
    out["modes"][mode] = {"headline": head, "mc": mcsum}
    out["mc_rows"][mode] = rows

with open(os.path.join(CONFIG["out_dir"], "results.json"), "w", encoding="utf-8") as f:
    json.dump(to_jsonable(out), f, ensure_ascii=False, indent=1)
csv_path = os.path.join(CONFIG["out_dir"], "mc_raw.csv")
with open(csv_path, "w", encoding="utf-8") as f:
    f.write("mode,rep," + ",".join(MC_KEYS) + "\n")
    for mode in CONFIG["modes_to_run"]:
        for i, row in enumerate(out["mc_rows"][mode]):
            f.write("%s,%d," % (mode, i) + ",".join(repr(row[k]) for k in MC_KEYS) + "\n")
print("已保存:", os.path.abspath(os.path.join(CONFIG["out_dir"], "results.json")))
print("已保存:", os.path.abspath(csv_path))

In [ ]:
# —— 蒙特卡洛汇总表 (对应文档表: 双重偏差与覆盖率) ——
# —— Monte Carlo summary table (corresponds to the document's table: double bias and coverage) ——
EST_CN = {"ols_base": "基准 OLS", "iv_clean": "2SLS 干净工具", "iv_dirty": "2SLS 脏工具",
          "did_implied": "DID 隐含 MPK", "did_fe": "FE 生产回归", "rdd": "RDD 局部估计"}
frames = []
for mode in CONFIG["modes_to_run"]:
    mc = out["modes"][mode]["mc"]
    for name in ["ols_base", "iv_clean", "iv_dirty", "did_implied", "did_fe", "rdd"]:
        it = mc[name]
        frames.append({
            "模式": MODE_CN[mode], "估计量": EST_CN[name],
            "均值": it["mean"], "SD": it["sd"],
            "2.5%": it["q025"], "97.5%": it["q975"],
            "自身识别对象": it["estimand"],
            "偏差vsμ_pop": it["bias_vs_pop"], "偏差vs识别对象": it["bias_vs_estimand"],
            "95%覆盖(μ_pop)": it["cover_pop"],
        })
df_mc = pd.DataFrame(frames)
pd.set_option("display.width", 160)
display(df_mc.round(4))

# —— 诊断指标 (对应文档"诊断全部通过"部分) ——
# —— diagnostics (corresponds to the document's "all diagnostics pass" section) ——
meta_rows = []
for mode in CONFIG["modes_to_run"]:
    m = out["modes"][mode]["mc"]["_meta"]
    meta_rows.append({
        "模式": MODE_CN[mode], "μ_pop": m["mu_pop"], "合规者平均": m["mu_comp"],
        "脏工具识别对象": m["ivd_estimand"], "DID隐含识别对象": m["did_implied_true"],
        "RDD局部真值": m["rdd_local_true"],
        "一阶段F(最小/中位)": "%.0f / %.0f" % (m["iv_F_min"], m["iv_F_med"]),
        "安慰剂覆盖0": m["did_placebo_cover0"],
        "RDD带宽均值(.5/1.5/2)": "%.3f/%.3f/%.3f" % (m["rdd_h05_mean"], m["rdd_h15_mean"], m["rdd_h20_mean"]),
    })
display(pd.DataFrame(meta_rows).round(4))

## 11. 图表
## 11. Figures

六张图与说明文档一致: 图1 三模式基准截面、图2 IV、图3 DID、图4 RDD、
图5 蒙特卡洛汇总、图6 "一个世界, 多个答案"。每张图**内联显示**并同时保存到
`humbug_output`。

Six figures matching the documentation: Fig. 1 baseline cross sections for the three
modes, Fig. 2 IV, Fig. 3 DID, Fig. 4 RDD, Fig. 5 Monte Carlo summary, Fig. 6
"one world, many answers". Each figure is **displayed inline** and simultaneously saved
to `humbug_output`.

In [ ]:
# 英文模式名映射 (仅新增, 不改动原有注释)
# English mode-name mapping (added only; existing comments unchanged)
MODE_EN = {"humbug": "Humbug (glyph data)",
           "neoclassical": "Neoclassical",
           "identity": "Accounting identity"}


def _save(fig, name):
    """保存到输出目录, 并在笔记本中内联显示 (Jupyter 适配)。
    Save to the output directory and display inline in the notebook (Jupyter-adapted)."""
    path = os.path.join(CONFIG["out_dir"], name)
    fig.savefig(path, dpi=300, facecolor="white")
    display(fig)
    plt.close(fig)
    print("  图已保存:", os.path.abspath(path))


def _dens(ax, x, label, color, ls, lw=1.6):
    """直方图密度曲线 (无 scipy 依赖)。
    Histogram density curve (no scipy dependency)."""
    xs = np.asarray(x, float)
    bins = np.linspace(xs.min() - 0.02, xs.max() + 0.02, 60)
    hist, edges = np.histogram(xs, bins=bins, density=True)
    mid = 0.5 * (edges[:-1] + edges[1:])
    ax.plot(mid, hist, ls, color=color, linewidth=lw, label=label,
            drawstyle="default", solid_capstyle="round")


def fig_baseline(out):
    """图1: 三种数据给定模式的基准截面与 OLS 拟合。
    Figure 1: baseline cross sections and OLS fits for the data-generating modes."""
    ms = CONFIG["modes_to_run"]                     # 支持子集模式 / supports a subset of modes (--modes)
    figw = 12.6 if len(ms) == 3 else 4.2 * len(ms)   # len=3 时与文档图严格同宽 / same width as the document figure when len=3
    fig, axes = plt.subplots(1, len(ms), figsize=(figw, 4.3),
                             constrained_layout=True, squeeze=False)
    for ax, mode in zip(axes[0], ms):
        kx, qy = get_letters()
        if mode == "humbug":
            x, y = kx, qy
        else:
            rng = np.random.default_rng([CONFIG["seed"], 99])
            fb = make_firms(mode, 1200, rng)
            x, y = fb["k0"], fb["q0"]
        ax.scatter(x, y, s=2.5, color=GK["d40"], alpha=0.55, linewidths=0)
        b = out["modes"][mode]["headline"]["baseline"]
        xr = np.linspace(x.min(), x.max(), 50)
        a0 = y.mean() - b["beta"] * x.mean()   # 过均值点的 OLS 拟合线 / OLS fit line through the mean point
        ax.plot(xr, a0 + b["beta"] * xr, "-", color=GK["black"], lw=1.4)
        ax.set_title("%s\nOLS slope = %.3f (SE %.3f), $R^2$ = %.3f"
                     % (MODE_EN[mode], b["beta"], b["se"], b["r2"]),
                     fontsize=10.5)
        ax.set_xlabel(r"$\ln(K/L)$  log capital per capita $k$", fontsize=9.5)
        ax.set_ylabel(r"$\ln(Y/L)$  log output per capita $q$", fontsize=9.5)
        ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
        ax.tick_params(labelsize=8.5)
    _save(fig, "fig1_baseline.png")


def fig_iv(out, dbg):
    """图2: IV 实验 — 第一阶段 / 单次估计森林 / MC 分布 (聚焦模式)。
    Figure 2: IV experiment — first stage / single-estimate forest / MC densities (focus mode)."""
    mode = CONFIG["focus_mode"]
    hd = out["modes"][mode]["headline"]
    mc = out["modes"][mode]["mc"]
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.3), constrained_layout=True)

    # (a) 第一阶段 / first stage
    ax = axes[0]
    d = dbg["iv"]
    ax.scatter(d["shock"]["Z"], d["k"], s=6, color=GK["d40"], alpha=0.5,
               linewidths=0)
    zs = np.linspace(-3, 3, 50)
    pi_hat = hd["iv_clean"]["pi_hat"]
    a0 = d["k"].mean() - pi_hat * d["shock"]["Z"].mean()   # 过均值点的第一阶段拟合线 / first-stage fit line through the mean point
    ax.plot(zs, a0 + pi_hat * zs, color=GK["black"], lw=1.5)
    ax.set_xlabel("Instrument $Z$ (credit supply shock)", fontsize=9.5)
    ax.set_ylabel(r"$\ln(K/L)$,  $k$", fontsize=9.5)
    ax.set_title("(a) First stage: $\\hat{\\pi}$ = %.3f (SE %.3f)\nF = %.0f (far above 10)"
                 % (pi_hat, hd["iv_clean"]["pi_se"], hd["iv_clean"]["first_stage_F"]),
                 fontsize=10.5)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)

    # (b) 单次估计森林图 / single-estimate forest plot
    ax = axes[1]
    rows = [("OLS (experiment sample)", hd["iv_clean"]["ols_beta"], hd["iv_clean"]["ols_se"]),
            ("2SLS clean instrument ($\\delta$=0)", hd["iv_clean"]["beta"], hd["iv_clean"]["se"]),
            ("2SLS dirty instrument ($\\delta$=0.08)", hd["iv_dirty"]["beta"], hd["iv_dirty"]["se"])]
    ys = np.arange(len(rows))[::-1]
    for (lbl, est, se), yy in zip(rows, ys):
        ax.plot([est - 1.96 * se, est + 1.96 * se], [yy, yy], color=GK["d60"],
                lw=1.2)
        ax.plot([est], [yy], "o", color=GK["black"], ms=6)
        ax.annotate("%.3f" % est, (est, yy), textcoords="offset points",
                    xytext=(8, 5), fontsize=8.5)
    ax.axvline(hd["iv_clean"]["mu_pop"], color=GK["black"], ls="--", lw=1.5)
    ax.axvline(hd["iv_clean"]["mu_compliers"], color=GK["d60"], ls=":", lw=1.5)
    ax.axvline(hd["iv_dirty"]["estimand"], color=GK["d60"], ls="-.", lw=1.4)
    ax.set_yticks(ys)
    ax.set_yticklabels([r[0] for r in rows], fontsize=9)
    ax.set_xlabel("Capital coefficient estimate (95% CI)", fontsize=9.5)
    ax.set_title("(b) Single estimate vs. the two benchmarks", fontsize=10.5)
    ax.grid(True, axis="x", alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(handles=[
        plt.Line2D([], [], color=GK["black"], ls="--", lw=1.5,
                   label="Population average true marginal product $\\mu_{pop}$"),
        plt.Line2D([], [], color=GK["d60"], ls=":", lw=1.5,
                   label="Complier average $E[\\mu\\pi]/E[\\pi]$"),
        plt.Line2D([], [], color=GK["d60"], ls="-.", lw=1.4,
                   label="Dirty-instrument estimand (incl. $\\delta$)"),
    ], loc="lower left", bbox_to_anchor=(0.0, 1.12), fontsize=8, frameon=False, ncol=1)

    # (c) MC 密度 / MC densities
    ax = axes[2]
    rows_c = out["mc_rows"][mode]
    _dens(ax, [r["iv_clean"] for r in rows_c], "2SLS clean instrument (500 reps)",
          GK["black"], "-")
    _dens(ax, [r["iv_dirty"] for r in rows_c], "2SLS dirty instrument (500 reps)",
          GK["d60"], "--")
    ax.axvline(mc["_meta"]["mu_pop"], color=GK["black"], ls="--", lw=1.5)
    ax.axvline(mc["_meta"]["mu_comp"], color=GK["d60"], ls=":", lw=1.5)
    ax.axvline(mc["_meta"]["ivd_estimand"], color=GK["d60"], ls="-.", lw=1.4)
    ax.set_xlabel("2SLS capital coefficient estimate", fontsize=9.5)
    ax.set_ylabel("Density", fontsize=9.5)
    ax.set_title("(c) Monte Carlo distribution (N = %d)" % CONFIG["mc_reps"], fontsize=10.5)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.14), fontsize=8, frameon=False)
    _save(fig, "fig2_iv.png")


def fig_did(out, dbg):
    """图3: DID 实验 — 平行趋势 / ATT 森林 / 隐含 MPK 的 MC 分布。
    Figure 3: DID experiment — parallel trends / ATT forest / implied-MPK MC densities."""
    mode = CONFIG["focus_mode"]
    hd = out["modes"][mode]["headline"]
    mc = out["modes"][mode]["mc"]
    panel = dbg["did"]
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4.3), constrained_layout=True)

    # (a) 平行趋势 / parallel trends
    ax = axes[0]
    T = panel["T"]
    tr, po = panel["treat"], panel["post"]
    mt = panel["q"][tr].mean(axis=0)
    mcq = panel["q"][~tr].mean(axis=0)
    ts = np.arange(T)
    ax.plot(ts, mt, "o-", color=GK["black"], lw=1.6, ms=5, label="Treated group (small firms)")
    ax.plot(ts, mcq, "s--", color=GK["d60"], lw=1.6, ms=5, mfc="white",
            label="Control group")
    ax.axvspan(po.argmax() - 0.5, T - 0.5, color=GK["d12"], alpha=0.6, zorder=0)
    ax.annotate("policy implemented →", xy=(0.985, 0.035), xycoords="axes fraction",
                ha="right", fontsize=8.5, color=GK["d60"])
    att = hd["did"]["theta"]
    ax.annotate("ATT = %.3f\n(true decomposition: %.3f·Δk + φ=%.3f)"
                % (att, hd["did"]["mu_treated"], hd["did"]["theta"] - CONFIG["did"]["phi"]),
                xy=(0.03, 0.96), xycoords="axes fraction", va="top", fontsize=8.5,
                bbox=dict(fc="white", ec=GK["d25"], lw=0.6, pad=3))
    ax.set_xlabel("Period $t$ (last two are post)", fontsize=9.5)
    ax.set_ylabel("Average $\\ln(Y/L)$", fontsize=9.5)
    ax.set_title("(a) Parallel trends holds by construction", fontsize=10.5)
    ax.set_xticks(ts)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.14), fontsize=8,
              frameon=False, ncol=2)

    # (b) 森林: 安慰剂 / ATT / FE / 隐含MPK / forest: placebo / ATT / FE / implied MPK
    ax = axes[1]
    dd = hd["did"]
    rows = [("Placebo pre-trend", hd["did_placebo"]["theta"], hd["did_placebo"]["se"]),
            ("DID policy effect ATT $\\hat{\\theta}$", dd["theta"], dd["se_theta"]),
            ("FE production regression $\\hat{\\beta}_{FE}$", dd["fe"], dd["se_fe"]),
            ("Implied marginal product $\\hat{\\theta}/\\Delta K$", dd["implied"], dd["implied_se"])]
    ys = np.arange(len(rows))[::-1]
    for (lbl, est, se), yy in zip(rows, ys):
        ax.plot([est - 1.96 * se, est + 1.96 * se], [yy, yy], color=GK["d60"], lw=1.2)
        ax.plot([est], [yy], "o", color=GK["black"], ms=6)
        ax.annotate("%.3f" % est, (est, yy), textcoords="offset points",
                    xytext=(8, 5), fontsize=8.5)
    ax.axvline(0.0, color=GK["d25"], lw=0.9)
    ax.axvline(dd["mu_pop"], color=GK["black"], ls="--", lw=1.5)
    ax.axvline(dd["mu_treated"], color=GK["d60"], ls=":", lw=1.5)
    ax.axvline(dd["implied_true"], color=GK["d60"], ls="-.", lw=1.4)
    ax.set_yticks(ys)
    ax.set_yticklabels([r[0] for r in rows], fontsize=9)
    ax.set_xlabel("Estimate (95% CI)", fontsize=9.5)
    ax.set_title("(b) Policy effect ≠ marginal product", fontsize=10.5)
    ax.grid(True, axis="x", alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(handles=[
        plt.Line2D([], [], color=GK["black"], ls="--", lw=1.5, label="$\\mu_{pop}$ population average"),
        plt.Line2D([], [], color=GK["d60"], ls=":", lw=1.5, label="Treated-group true mean $\\mu$"),
        plt.Line2D([], [], color=GK["d60"], ls="-.", lw=1.4, label="Implied-MPK estimand"),
    ], loc="lower left", bbox_to_anchor=(0.0, 1.12), fontsize=8, frameon=False)

    # (c) MC 密度: 隐含 MPK / MC densities: implied MPK
    ax = axes[2]
    rows_c = out["mc_rows"][mode]
    _dens(ax, [r["did_implied"] for r in rows_c],
          "Implied marginal product $\\hat{\\theta}/\\Delta K$ (500 reps)",
          GK["black"], "-")
    _dens(ax, [r["did_fe"] for r in rows_c], "FE production regression (500 reps)",
          GK["d60"], "--")
    ax.axvline(mc["_meta"]["mu_pop"], color=GK["black"], ls="--", lw=1.5)
    ax.axvline(mc["_meta"]["did_implied_true"], color=GK["d60"], ls="-.", lw=1.4)
    ax.set_xlabel("DID-implied marginal product of capital", fontsize=9.5)
    ax.set_ylabel("Density", fontsize=9.5)
    ax.set_title("(c) Monte Carlo distribution (N = %d)" % CONFIG["mc_reps"], fontsize=10.5)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.14), fontsize=8, frameon=False)
    _save(fig, "fig3_did.png")


def fig_rdd(out, dbg):
    """图4: RDD 实验 — 资本跳跃 / 产出跳跃 / 异质性扫描 / 外推误差。
    Figure 4: RDD experiment — capital jump / output jump / heterogeneity scan / extrapolation error."""
    mode = CONFIG["focus_mode"]
    hd = out["modes"][mode]["headline"]
    data = dbg["rdd"]
    s, k, q, D = data["s"], data["k"], data["q"], data["D"]
    h = CONFIG["rdd"]["h"]
    fig, axes = plt.subplots(2, 2, figsize=(10.8, 8.0), constrained_layout=True)

    def binned(ax, y, ylabel, title, est_jump):
        edges = np.linspace(-CONFIG["rdd"]["s_max"], CONFIG["rdd"]["s_max"], 41)
        mids = 0.5 * (edges[:-1] + edges[1:])
        vals = []
        for i in range(40):
            msk = (s >= edges[i]) & (s < edges[i + 1])
            vals.append(y[msk].mean() if msk.any() else np.nan)
        ax.plot(mids, vals, "o", color=GK["d40"], ms=3.5, mfc="white", lw=0)
        # 局部线性拟合 (窗口内) / local-linear fit (within window)
        m = np.abs(s) <= h
        for side, msk, ls in ((0, m & (D == 0), "-"), (1, m & (D == 1), "--")):
            xs, ys_ = s[msk], y[msk]
            bb = np.polyfit(xs, ys_, 1)
            xr = np.linspace(-h if side == 0 else 0.0, 0.0 if side == 0 else h, 10)
            ax.plot(xr, np.polyval(bb, xr), ls, color=GK["black"], lw=1.5)
        ax.axvline(0.0, color=GK["d25"], lw=0.9, ls=":")
        ax.annotate("Jump = %.3f" % est_jump, xy=(0.04, 0.5), xycoords="axes fraction",
                    fontsize=9, bbox=dict(fc="white", ec=GK["d25"], lw=0.6, pad=3))
        ax.set_xlabel("Firm size $s$ (threshold $s^*=0$)", fontsize=9.5)
        ax.set_ylabel(ylabel, fontsize=9.5)
        ax.set_title(title, fontsize=10.5)
        ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
        ax.tick_params(labelsize=8.5)

    binned(axes[0, 0], k, r"$\ln(K/L)$,  $k$",
           "(a) Capital jumps at the threshold, $J_k$ = %.3f"
           % hd["rdd"]["Jk"], hd["rdd"]["Jk"])
    binned(axes[0, 1], q, r"$\ln(Y/L)$,  $q$",
           "(b) Output jumps at the threshold, $J_q$ = %.3f"
           % hd["rdd"]["Jq"], hd["rdd"]["Jq"])

    # (c) 伪门槛扫描 vs 真实异质性 / pseudo-cutoff scan vs true heterogeneity
    ax = axes[1, 0]
    sg = np.linspace(-2.4, 2.4, 200)
    ax.plot(sg, true_mu_curve(mode, sg), "-", color=GK["black"], lw=1.6,
            label="True $E[\\mu|s]$ (simulator view)")
    ps = hd["rdd_pseudo"]
    ax.plot([p["cutoff"] for p in ps], [p["beta"] for p in ps], "s",
            color=GK["d60"], ms=6, mfc="white", label="RDD estimate at placebo cutoff")
    ax.plot([0.0], [hd["rdd"]["beta"]], "o", color=GK["black"], ms=8,
            label="Estimate at true cutoff ($s^*=0$)")
    ax.axhline(hd["extrapolation"]["mu_pop"], color=GK["black"], ls="--", lw=1.5,
               label="Population average $\\mu_{pop}$")
    ax.set_xlabel("Policy cutoff location $s^*$", fontsize=9.5)
    ax.set_ylabel("RDD-estimated local marginal product", fontsize=9.5)
    ax.set_title("(c) Local estimates shift with the cutoff", fontsize=10.5)
    ax.grid(True, alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 1.16), fontsize=8, frameon=False)

    # (d) 外推误差 / extrapolation error
    ax = axes[1, 1]
    ex = hd["extrapolation"]
    bars = [("RDD local estimate, extrapolated\n$\\hat{\\beta}_{RDD}\\times\\gamma$",
             ex["pred"], GK["d60"]),
            ("True total effect\n$\\mu_{pop}\\times\\gamma$", ex["true"], GK["black"])]
    for i, (lbl, val, cc) in enumerate(bars):
        ax.bar(i, val, width=0.5, color=cc, alpha=0.85 if i else 0.55,
               edgecolor=GK["black"], lw=0.8)
        ax.annotate("%.3f" % val, (i, val), ha="center", va="bottom", fontsize=9.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([b[0] for b in bars], fontsize=8.8)
    ax.set_ylabel("Output-per-capita growth when all firms raise capital by $\\gamma$",
                  fontsize=9.5)
    ax.set_title("(d) Extrapolating local to aggregate: %.0f%% underestimate"
                 % (abs(ex["rel_error"]) * 100), fontsize=10.5)
    ax.grid(True, axis="y", alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=8.5)
    ax.set_ylim(0, max(ex["pred"], ex["true"]) * 1.22)
    _save(fig, "fig4_rdd.png")


def fig_mc(out):
    """图5: 蒙特卡洛汇总 — 三模式下五种估计量的分布 (N=500)。
    Figure 5: Monte Carlo summary — distributions of five estimators across the three modes (N=500)."""
    ms = CONFIG["modes_to_run"]
    figw = 12.6 if len(ms) == 3 else 4.2 * len(ms)
    fig, axes = plt.subplots(1, len(ms), figsize=(figw, 4.5),
                             constrained_layout=True, squeeze=False)
    labels = ["OLS", "2SLS\nclean", "2SLS\ndirty", "DID\nimplied", "RDD"]
    keys = ["ols_base", "iv_clean", "iv_dirty", "did_implied", "rdd"]
    est_keys = ["mu_pop", "mu_comp", "ivd_estimand", "did_implied_true",
                "rdd_local_true"]
    for ax, mode in zip(axes[0], ms):
        rows = out["mc_rows"][mode]
        data = [[r[k] for r in rows] for k in keys]
        bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False,
                        medianprops=dict(color=GK["black"], lw=1.4),
                        whiskerprops=dict(color=GK["d60"], lw=1.0),
                        capprops=dict(color=GK["d60"], lw=1.0),
                        boxprops=dict(facecolor=GK["d12"], edgecolor=GK["d60"], lw=0.9))
        meta = out["modes"][mode]["mc"]["_meta"]
        ax.axhline(meta["mu_pop"], color=GK["black"], ls="--", lw=1.5)
        for i, ek in enumerate(est_keys):
            ax.plot([i + 1], [meta[ek]], "x", color=GK["d60"], ms=7, mew=1.8,
                    zorder=5)
        ax.set_xticks(range(1, 6))
        ax.set_xticklabels(labels, fontsize=8.6)
        ax.set_title(MODE_EN[mode] + "  ($\\mu_{pop}$ = %.3f)" % meta["mu_pop"],
                     fontsize=10.5)
        ax.grid(True, axis="y", alpha=0.25, color=GK["d25"], lw=0.5)
        ax.tick_params(labelsize=8.5)
    handles = [plt.Line2D([], [], color=GK["black"], ls="--", lw=1.5,
                          label="Population average true marginal product $\\mu_{pop}$"),
               plt.Line2D([], [], color=GK["d60"], marker="x", ls="", ms=7, mew=1.8,
                          label="Method-specific estimand")]
    fig.legend(handles=handles, loc="outside lower center", ncol=2, fontsize=9,
               frameon=False)
    _save(fig, "fig5_mc.png")


def fig_answers(out, dbg):
    """图6: 一个世界, 多个答案 — 聚焦模式下全部估计量与置信区间。
    Figure 6: one world, many answers — all estimators and confidence intervals in the focus mode."""
    mode = CONFIG["focus_mode"]
    hd = out["modes"][mode]["headline"]
    rows = [
        ("Baseline OLS (no policy cross-section)", hd["baseline"]["beta"], hd["baseline"]["se"],
         hd["extrapolation"]["mu_pop"]),
        ("OLS (IV experiment sample)", hd["iv_clean"]["ols_beta"], hd["iv_clean"]["ols_se"],
         hd["iv_clean"]["mu_pop"]),
        ("2SLS clean instrument", hd["iv_clean"]["beta"], hd["iv_clean"]["se"],
         hd["iv_clean"]["mu_compliers"]),
        ("2SLS dirty instrument", hd["iv_dirty"]["beta"], hd["iv_dirty"]["se"],
         hd["iv_dirty"]["estimand"]),
        ("DID-implied marginal product $\\hat{\\theta}/\\Delta K$", hd["did"]["implied"], hd["did"]["implied_se"],
         hd["did"]["implied_true"]),
        ("FE production regression", hd["did"]["fe"], hd["did"]["se_fe"],
         hd["did"]["implied_true"]),
        ("RDD local estimate ($s^*=0$)", hd["rdd"]["beta"], hd["rdd"]["se"],
         hd["rdd_local_true"]),
    ]
    fig, ax = plt.subplots(figsize=(9.6, 5.4), constrained_layout=True)
    ys = np.arange(len(rows))[::-1]
    for (lbl, est, se, estand), yy in zip(rows, ys):
        ax.plot([est - 1.96 * se, est + 1.96 * se], [yy, yy], color=GK["d60"], lw=1.3)
        ax.plot([est], [yy], "o", color=GK["black"], ms=7, zorder=5)
        ax.plot([estand], [yy], "x", color=GK["d60"], ms=8, mew=1.9, zorder=5)
        ax.annotate("%.3f" % est, (est, yy), textcoords="offset points",
                    xytext=(9, 4), fontsize=9)
    ax.axvline(hd["extrapolation"]["mu_pop"], color=GK["black"], ls="--", lw=1.6)
    ax.set_yticks(ys)
    ax.set_yticklabels([r[0] for r in rows], fontsize=9.5)
    ax.set_xlabel("Capital coefficient / implied marginal product estimate (95% CI)",
                  fontsize=10)
    ax.set_title("%s: \"marginal product of capital\" from seven conventional procedures"
                 % MODE_EN[mode], fontsize=11.5)
    ax.grid(True, axis="x", alpha=0.25, color=GK["d25"], lw=0.5)
    ax.tick_params(labelsize=9)
    handles = [plt.Line2D([], [], marker="o", color=GK["black"], ls="", ms=7,
                          label="Point estimate (●) with 95% CI"),
               plt.Line2D([], [], color=GK["black"], ls="--", lw=1.6,
                          label="Population average true marginal product $\\mu_{pop}$"),
               plt.Line2D([], [], color=GK["d60"], marker="x", ls="", ms=8, mew=1.9,
                          label="Method-specific estimand (×)")]
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.03, 1.0),
              fontsize=9, frameon=False, labelspacing=1.1)
    _save(fig, "fig6_answers.png")

In [ ]:
fig_baseline(out)
if CONFIG["focus_mode"] in debugs:
    fig_iv(out, debugs[CONFIG["focus_mode"]])
    fig_did(out, debugs[CONFIG["focus_mode"]])
    fig_rdd(out, debugs[CONFIG["focus_mode"]])
    fig_answers(out, debugs[CONFIG["focus_mode"]])
fig_mc(out)
print("全部图表完成。")

## 12. 导出 Excel 汇总
## 12. Export Excel summary

把蒙特卡洛汇总、单次结果与参数配置写入 `humbug_output/MC汇总表_notebook.xlsx`
(与随文档交付的 `MC_Notebook.xlsx` 同构; 需要 `openpyxl`, Anaconda 自带)。

Write the Monte Carlo summary, single-run results, and parameter configuration to
`humbug_output/MC汇总表_notebook.xlsx` (isomorphic to the `MC_Notebook.xlsx` shipped
with the documentation; requires `openpyxl`, bundled with Anaconda).

In [ ]:
xlsx_path = os.path.join(CONFIG["out_dir"], "MC_notebook.xlsx")
try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
        df_mc.round(4).to_excel(xw, sheet_name="MC汇总", index=False)
        df_single.round(4).to_excel(xw, sheet_name="单次结果", index=False)
        pd.DataFrame([{"参数": k, "取值": repr(v)}
                      for k, v in CONFIG.items() if k != "out_dir"]
                     ).to_excel(xw, sheet_name="参数", index=False)
    print("已导出:", os.path.abspath(xlsx_path))
except ImportError:
    print("缺少 openpyxl, 跳过 Excel 导出: conda install openpyxl 后重跑本单元格即可。")

## 结果要点 (种子 2026, 与说明文档一致)
## Key results (seed 2026, consistent with the documentation)

1. **HUMBUG 世界**: 基准 OLS 失效 (β ≈ 0.02, R² ≈ 0.001); 但第一阶段 F ≈ 545、
   安慰剂检验通过、RDD 带宽与伪门槛诊断正常 —— 常规诊断全部通过, 而
   干净 IV ≈ 0.37 (合规者 LATE)、脏 IV ≈ 0.56、DID 隐含 ≈ 0.50、RDD ≈ 0.13 (门槛局部),
   均不等于 μ_pop = 0.30。
2. **新古典世界**: 干净 IV 与 RDD 精确复原 0.30 (95% 覆盖率 ≈ 0.95) ——
   说明 HUMBUG 世界中的失败不是程序错误, 而是"识别对象 ≠ 理论构念"的错位。
3. **恒等式世界**: 与新古典世界逐值相同 (基准 R² = 1.000) ——
   因果识别程序无法区分"真生产函数"与"会计恒等式"。

1. **HUMBUG world**: baseline OLS fails (β ≈ 0.02, R² ≈ 0.001); yet the first-stage
   F ≈ 545, the placebo test passes, and RDD bandwidth / pseudo-cutoff diagnostics look
   fine — all conventional diagnostics pass, while clean IV ≈ 0.37 (complier LATE),
   dirty IV ≈ 0.56, DID-implied ≈ 0.50, RDD ≈ 0.13 (local at the threshold) — none of
   which equals μ_pop = 0.30.
2. **Neoclassical world**: clean IV and RDD recover 0.30 exactly (95% coverage ≈ 0.95) —
   showing that the failure in the HUMBUG world is not a coding error but a mismatch
   between "what is identified" and "the theoretical construct".
3. **Identity world**: numerically identical to the neoclassical world (baseline
   R² = 1.000) — causal-identification procedures cannot distinguish a "true production
   function" from an "accounting identity".

> **复现提示 / Reproduction note**: 修改 `CONFIG["mc_reps"]`、`CONFIG["focus_mode"]`、
> `CONFIG["modes_to_run"]` 或任一结构参数 (δ、φ、异质性系数) 后,
> 从第 9/10 节起重跑即可; 第 1–8 节的函数定义无需重复执行。
>
> After changing `CONFIG["mc_reps"]`, `CONFIG["focus_mode"]`, `CONFIG["modes_to_run"]`,
> or any structural parameter (δ, φ, heterogeneity coefficients), just re-run from
> Sections 9/10 onward; the function definitions in Sections 1–8 need not be re-executed.